## Project 2 — Healthcare Prediction & Patient Assistance System
## Exploratory Data Analysis (EDA) & Machine Learning Model Evaluation

This notebook documents the exploratory data analysis, class balance checking, and multi-model evaluation (Logistic Regression, Random Forest, and Gradient Boosting) performed for the capstone project.

In [16]:
import sys
!{sys.executable} -m pip install scikit-learn


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('diabetes_prediction_dataset.csv')
print('Dataset Shape:', df.shape)
df.head()

Dataset Shape: (100000, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


### 1. Data Quality & Basic Info
Checking data types, missing values, and statistical summaries.

In [18]:
print('--- Dataset Info ---')
print(df.info())

print('\n--- Missing Values ---')
print(df.isnull().sum())

print('\n--- Statistical Summary ---')
print(df.describe())

--- Dataset Info ---
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  str    
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  str    
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 6.9 MB
None

--- Missing Values ---
gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: i

### 2. Class Balance Analysis
Checking the distribution of the target variable (`diabetes`) using tabular summaries.

In [19]:
counts = df['diabetes'].value_counts()
percentages = df['diabetes'].value_counts(normalize=True) * 100

print('Class Distribution Counts:')
print(counts)
print('\nClass Proportions (%):')
print(percentages)

Class Distribution Counts:
diabetes
0    91500
1     8500
Name: count, dtype: int64

Class Proportions (%):
diabetes
0    91.5
1     8.5
Name: proportion, dtype: float64


### 3. Model Training & Comparison
Splitting the data, building preprocessing pipelines, and comparing multiple algorithms across performance metrics.

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

# Features and Target
X = df.drop(columns=['diabetes'])
y = df['diabetes']

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Engineering / Preprocessing
numeric_features = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'hypertension', 'heart_disease']
categorical_features = ['gender', 'smoking_history']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

# Dictionary of models to compare
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

# Training and Evaluation Loop
results = {}
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4)
    }

# Display Performance Comparison Table
performance_df = pd.DataFrame(results).T
print('Model Evaluation Comparison:')
print(performance_df)

Model Evaluation Comparison:
                     Accuracy  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression    0.9605     0.8593  0.6394    0.7332   0.9625
Random Forest          0.9700     0.9393  0.6924    0.7972   0.9627
Gradient Boosting      0.9724     0.9807  0.6888    0.8093   0.9791
